# EDA: shapfeatures.csv

This notebook loads `shapfeatures.csv` and runs a quick exploratory data analysis:

- Basic shape and preview
- Missingness and duplicates
- `label` and `dataset_id` distributions
- Numeric feature summaries
- Top features by mean absolute value
- Distribution snapshots
- Correlation heatmap (top features)
- PCA 2D projection (colored by label if available)
- Simple label association proxy (variability of class means)


In [ ]:
# Imports + load data
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

csv_path = './shapfeatures.csv'
df_shap = pd.read_csv(csv_path, encoding='ascii')

df_shap.head()

In [ ]:
# Basic overview: shape, columns, duplicates, missingness
print(df_shap.shape)
print(df_shap.columns[:20])
print(df_shap.columns[-10:])

n_dupes = df_shap.duplicated().sum()
print(n_dupes)

missing_counts = df_shap.isna().sum().sort_values(ascending=False)
missing_pct = (missing_counts / len(df_shap) * 100).round(2)
missing_summary = pd.DataFrame({'missing_count': missing_counts, 'missing_pct': missing_pct})
missing_summary.head(20)

In [ ]:
# Target and cohort distributions
if 'label' in df_shap.columns:
    print(df_shap['label'].value_counts(dropna=False).head(20))
    plt.figure(figsize=(8,4))
    sns.countplot(data=df_shap, x='label', order=df_shap['label'].value_counts().index)
    plt.title('Label distribution')
    plt.xticks(rotation=45, ha='right')
    plt.tight_layout()
    plt.show()

if 'dataset_id' in df_shap.columns:
    print(df_shap['dataset_id'].value_counts(dropna=False).head(20))

In [ ]:
# Numeric feature columns + summary stats
numeric_cols = df_shap.select_dtypes(include=[np.number]).columns.tolist()
non_feature_cols = set([c for c in ['Unnamed: 0', 'label', 'dataset_id'] if c in df_shap.columns])
feature_cols = [c for c in numeric_cols if c not in non_feature_cols]

print('n_numeric_cols')
print(len(numeric_cols))
print('n_feature_cols')
print(len(feature_cols))

# Full describe is big; compute and sort by mean absolute value
if len(feature_cols) > 0:
    desc = df_shap[feature_cols].describe(percentiles=[0.01, 0.05, 0.5, 0.95, 0.99]).T
    desc['missing'] = df_shap[feature_cols].isna().sum()
    desc['missing_pct'] = (desc['missing'] / len(df_shap) * 100).round(2)
    desc['mean_abs'] = df_shap[feature_cols].abs().mean()
    desc_sorted = desc.sort_values('mean_abs', ascending=False)
    desc_sorted.head(15)

In [ ]:
# Plot: Top features by mean absolute value + missingness
if len(feature_cols) > 0:
    top_n = min(30, len(feature_cols))
    top_features = desc_sorted.head(top_n).index.tolist()

    plt.figure(figsize=(10,6))
    sns.barplot(x=desc_sorted.loc[top_features, 'mean_abs'].values, y=top_features, color='#2b8cbe')
    plt.title('Top features by mean(|value|)')
    plt.xlabel('Mean absolute value')
    plt.ylabel('Feature')
    plt.tight_layout()
    plt.show()

    top_missing = missing_summary.loc[feature_cols].sort_values('missing_pct', ascending=False).head(25)
    if len(top_missing) > 0 and top_missing['missing_count'].max() > 0:
        plt.figure(figsize=(10,6))
        sns.barplot(x=top_missing['missing_pct'].values, y=top_missing.index.tolist(), color='#7bccc4')
        plt.title('Top 25 features by missingness (%)')
        plt.xlabel('Missing %')
        plt.ylabel('Feature')
        plt.tight_layout()
        plt.show()

In [ ]:
# Distribution snapshots for a few top features
if len(feature_cols) > 0:
    if 'top_features' in locals():
        plot_features = top_features[:6]
    else:
        plot_features = feature_cols[:6]

    fig, axes = plt.subplots(2, 3, figsize=(12,7))
    axes = axes.flatten()

    for i, col in enumerate(plot_features):
        ax = axes[i]
        vals = df_shap[col]
        sns.histplot(vals, bins=50, kde=True, ax=ax, color='#feb24c')
        ax.set_title(col)

    for j in range(i + 1, len(axes)):
        axes[j].axis('off')

    plt.suptitle('Feature distribution snapshots', y=1.02)
    plt.tight_layout()
    plt.show()

In [ ]:
# Correlation heatmap (top 25 features by mean(|value|))
if len(feature_cols) > 2:
    if 'top_features' in locals():
        corr_features = top_features[:25]
    else:
        corr_features = feature_cols[:25]

    corr_mat = df_shap[corr_features].corr(numeric_only=True)
    plt.figure(figsize=(10,8))
    sns.heatmap(corr_mat, cmap='vlag', center=0, linewidths=0.2)
    plt.title('Correlation heatmap (top 25 by mean(|value|))')
    plt.tight_layout()
    plt.show()

In [ ]:
# PCA 2D projection (quick look)
from sklearn.decomposition import PCA
from sklearn.preprocessing import StandardScaler

if len(feature_cols) > 3:
    X = df_shap[feature_cols].copy().fillna(0.0)
    X_scaled = StandardScaler(with_mean=True, with_std=True).fit_transform(X.values)

    pca = PCA(n_components=2, random_state=0)
    pcs = pca.fit_transform(X_scaled)

    df_pca = pd.DataFrame({'PC1': pcs[:, 0], 'PC2': pcs[:, 1]})
    if 'label' in df_shap.columns:
        df_pca['label'] = df_shap['label'].astype(str).values

    plt.figure(figsize=(8,6))
    if 'label' in df_pca.columns:
        sns.scatterplot(data=df_pca, x='PC1', y='PC2', hue='label', s=35, alpha=0.8)
        plt.legend(bbox_to_anchor=(1.02, 1), loc='upper left', borderaxespad=0)
    else:
        sns.scatterplot(data=df_pca, x='PC1', y='PC2', s=35, alpha=0.8)

    plt.title('PCA projection (2 components)')
    plt.tight_layout()
    plt.show()

    print(pca.explained_variance_ratio_)
    print(float(pca.explained_variance_ratio_.sum()))

In [ ]:
# Simple label association proxy: variability of class means
if 'label' in df_shap.columns and len(feature_cols) > 0:
    grp_means = df_shap.groupby('label')[feature_cols].mean(numeric_only=True)
    class_mean_std = grp_means.std(axis=0).sort_values(ascending=False)

    top_sep = class_mean_std.head(20)
    print(top_sep)

    plt.figure(figsize=(10,6))
    sns.barplot(x=top_sep.values, y=top_sep.index.tolist(), color='#8856a7')
    plt.title('Top 20 features by variability of class means (proxy for separability)')
    plt.xlabel('Std dev of class means')
    plt.ylabel('Feature')
    plt.tight_layout()
    plt.show()

    show_cols = top_sep.index.tolist()[:8]
    grp_means[show_cols]